# Retrieval-Augmented Generation with Vincio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohswedd/vincio/blob/main/examples/notebooks/02_rag.ipynb)

This tutorial builds a **grounded, cited, eval-scored** question answerer over your own
documents using [Vincio](https://github.com/Ohswedd/vincio) — a Python context-engineering
platform whose single entry point is `from vincio import ContextApp`.

You will see two equivalent paths to the same governed run:

1. **The one-line front door** — `rag(<dir>).ask("...")`
2. **The verbose `ContextApp` path** — `add_source` + `set_policy` + `add_evaluator` + `run`

Then we inspect the **citations** and **eval scores** stamped on every answer, and finish
with metadata-filtered / hybrid retrieval. Everything runs **fully offline** — no API keys,
no network — on the bundled deterministic mock provider.

In [ ]:
!pip install -q vincio

## Setup — an offline-by-default provider

A bare `ContextApp()` defaults to OpenAI and needs an API key. To stay offline we use the
bundled **mock provider**, which returns deterministic output and — importantly for RAG —
still produces **real citations, traces, and cost** from the grounding pipeline.

The `_provider()` helper below is the standard Vincio pattern: it returns the mock by
default, and a real provider when you set environment variables.

> **To run against a real model**, set `VINCIO_PROVIDER` (e.g. `openai`) and the matching
> API key (e.g. `OPENAI_API_KEY`); optionally set `VINCIO_MODEL`. The code below is
> unchanged — only the environment differs.

In [ ]:
import os
import tempfile
from pathlib import Path

from vincio import ContextApp, rag
from vincio.providers import MockProvider, build_provider


def _provider():
    """Offline mock by default; a real provider when VINCIO_PROVIDER is set."""
    name = os.environ.get("VINCIO_PROVIDER", "mock")
    if name == "mock":
        return MockProvider(), "mock-1"
    return build_provider(name), os.environ.get("VINCIO_MODEL", "gpt-4o-mini")

## A tiny knowledge base

RAG grounds answers in *your* documents. We write a couple of small Markdown files to a
temporary directory so the whole notebook is self-contained — nothing touches your project.

In [ ]:
docs_dir = Path(tempfile.mkdtemp()) / "kb"
docs_dir.mkdir(parents=True, exist_ok=True)

(docs_dir / "refund_policy.md").write_text(
    "# Refund Policy\n\n"
    "Customers on the Pro plan may request refunds within 30 days of purchase "
    "with no processing fee.\n\n"
    "Basic plan refunds incur a $5 fee and must be requested within 14 days.\n",
    encoding="utf-8",
)
(docs_dir / "terms.md").write_text(
    "# Terms of Service\n\n"
    "The subscription renews automatically unless terminated 60 days before the "
    "renewal date. The initial term is 24 months.\n",
    encoding="utf-8",
)

print("wrote:", sorted(p.name for p in docs_dir.iterdir()))

## (A) The one-line front door — `rag(...).ask(...)`

`rag(sources)` indexes your sources, turns on **grounding-only** answering, and adds the
**groundedness** + **citation-accuracy** evaluators — the canonical six-call RAG path as a
single expression. `.ask(question)` runs a full grounded, cited, eval-scored governed run
and returns a `RunResult`.

`rag(...)` is `@experimental`, so you may see a `VincioExperimentalWarning` — that is
expected, not an error.

In [ ]:
answer = rag(str(docs_dir), provider=_provider()[0], model=_provider()[1],
             chunking="adaptive").ask("What is the refund window for the Pro plan?")

print("answer   :", answer.output)
print("citations:", answer.citations)
print("eval     :", answer.eval_scores)

# On the offline mock the answer text is a placeholder, but the citations, eval scores,
# trace_id, and cost are all real outputs of the grounding pipeline.

## (B) The equivalent verbose `ContextApp` path

The front door is sugar. Here is the explicit, six-call form it compiles to:

- `add_source(...)` indexes a folder (adaptive chunking + hybrid retrieval)
- `set_policy("answer_only_from_sources", True)` forbids ungrounded answers
- `add_evaluator("groundedness")` / `add_evaluator("citation_accuracy")` score every run
- `run(question)` executes the governed turn

This **lowers to the same governed run** as `rag(...).ask(...)` above — same retrieval,
same grounding policy, same evaluators, same compiled context packet.

In [ ]:
provider, model = _provider()
app = ContextApp(name="docs_qa", provider=provider, model=model)

# 1. Index the same folder.
app.add_source("kb", path=str(docs_dir), chunking="adaptive", retrieval="hybrid")

# 2. Grounding-only: the model may answer solely from retrieved evidence.
app.set_policy("answer_only_from_sources", True)

# 3. Measure groundedness + citation accuracy on every run.
app.add_evaluator("groundedness")
app.add_evaluator("citation_accuracy")

# 4. Run the governed turn.
result = app.run("What is the refund window for the Pro plan?")

print("answer   :", result.output)
print("citations:", result.citations)
print("eval     :", result.eval_scores)

## (C) Inspecting citations and eval scores

Every `RunResult` is fully observable. `result.citations` reports exactly which evidence
chunks backed the answer; `result.eval_scores` carries the metric verdicts; and each run is
stamped with a `trace_id` and a `cost_usd` for audit and accounting.

In [ ]:
print("trace_id :", result.trace_id)
print(f"cost_usd : ${result.cost_usd:.6f}")
print(f"tokens   : {result.usage.total_tokens} total")
print()

print("Citations backing the answer:")
for i, citation in enumerate(result.citations, 1):
    print(f"  [{i}] {citation}")

print()
print("Eval scores (metric -> score):")
for metric, score in result.eval_scores.items():
    print(f"  {metric:<20} {score}")

## (D) Metadata-filtered & hybrid retrieval

Relevance is not the only constraint. Vincio's retrieval layer fuses complementary signals
(BM25 lexical + dense semantic + more) and lets you push **hard metadata filters** into the
search with a serializable `FilterSpec` — useful for tenancy, document kind, or field
values. The example below builds two chunks tagged by `plan` and retrieves *only* the Pro
plan chunk, fully offline.

> The verbose `app.add_source(..., retrieval="hybrid")` you used above already runs hybrid
> keyword+vector retrieval; here we drop one level lower to show the filter pushdown.

In [ ]:
import asyncio

from vincio.core.types import Chunk
from vincio.retrieval import BM25Index, FilterSpec, and_, eq


async def filtered_retrieval():
    chunks = [
        Chunk(document_id="kb", index=0,
              text="Pro plan refunds within 30 days, no fee.",
              metadata={"plan": "Pro", "category": "policy"}),
        Chunk(document_id="kb", index=1,
              text="Basic plan refunds within 14 days with a $5 fee.",
              metadata={"plan": "Basic", "category": "policy"}),
    ]
    index = BM25Index()
    await index.add(chunks)

    # A serializable boolean filter tree — round-trips as JSON and compiles to
    # native predicates for hosted vector stores (Pinecone, pgvector, ...).
    spec = and_(eq("plan", "Pro"), eq("category", "policy"))
    revived = FilterSpec.model_validate_json(spec.model_dump_json())  # lossless round-trip

    hits = await index.search("refunds", top_k=5, where=revived)
    print(f"plan=Pro filter -> {len(hits)} hit(s)")
    for hit in hits:
        print("  ", hit.chunk.text)


asyncio.run(filtered_retrieval())

## Recap

- `rag(<dir>).ask("...")` is the one-line front door to grounded, cited, eval-scored RAG.
- It lowers to the verbose `add_source` + `set_policy("answer_only_from_sources")` +
  `add_evaluator(...)` + `run(...)` path — same governed run, your choice of altitude.
- Every answer carries `citations`, `eval_scores`, a `trace_id`, and a `cost_usd`.
- Hybrid retrieval and serializable `FilterSpec` metadata pushdown give you precise control
  over what evidence is eligible.

Swap in a real model any time by setting `VINCIO_PROVIDER` and the matching API key — the
code does not change.